## 第2周 第2天

我们的第一个 Agentic 框架项目！！

准备好，这将非常简单。

我们将构建一个简单的 Agent 系统，用于生成冷销售外发邮件：
1. Agent 工作流
2. 使用工具调用函数
3. 通过 Tools 和 Handoffs 实现 Agent 协作

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">重要提示 - 使用 SendGrid 发送邮件</h2>
            <span style="color:#ff7800;">我们将使用邮件服务商 SendGrid。但这是可选的。<br/>
            下一个单元格包含 SendGrid 的设置说明。但如果遇到问题，或者你想使用其他替代方案，<br/>
            请查看 <a href="https://edwarddonner.com/faq">FAQ 页面的 Q29</a> 获取完整说明和替代方案。
            </span>
        </td>
    </tr>
</table>

## 设置 SendGrid


请访问 Sendgrid：https://sendgrid.com/

（Sendgrid 是 Twilio 旗下的邮件发送服务公司。）

__如果 SendGrid 遇到问题，请查看 FAQ 页面 https://edwarddonner.com/faq 的 Q29 中的替代方案，包括 community_contributions/2_lab2_with_resend_email 中的 "Resend Email" 方案，或者直接跳过邮件部分。__

注册 SendGrid 账户是免费的！（至少目前对我来说是这样。）

创建账户后，点击：

Settings（左侧边栏）>> API Keys >> Create API Key（右上角按钮）

复制密钥到剪贴板，然后在你的 .env 文件中添加一行：

`SENDGRID_API_KEY=xxxx`

另外，在 SendGrid 中，前往：

Settings（左侧边栏）>> Sender Authentication >> "Verify a Single Sender"  
验证你自己的邮箱地址是真实的，这样 SendGrid 才能为你发送邮件。

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import resend
import os
import asyncio



In [ ]:
load_dotenv(override=True)

In [ ]:
# 让我们检查 Resend 邮件功能是否正常工作

def send_test_email():
    import resend
    resend.api_key = 're_QTqQh9Yx_6ceG31HmFphEhpSncDhgBrZh'
    params = {
        'from': 'test@ztest.online',
        'to': ['francullimash@gmail.com'],
        'subject': 'Test email from Resend',
        'text': 'This is an important test email',
    }
    email = resend.Emails.send(params)
    print(email.get('id'))

send_test_email()

### 你收到测试邮件了吗

如果返回 202，说明一切就绪！

#### 证书错误

如果遇到 SSL: CERTIFICATE_VERIFY_FAILED 错误，学生 Chris S 和 Oleksandr K 提供了以下建议：  
首先运行：`!uv pip install --upgrade certifi`  
然后运行：
```python
import certifi
import os
os.environ['SSL_CERT_FILE'] = certifi.where()
```

#### 其他错误或未收到邮件

如果遇到其他问题，你需要检查 SendGrid 控制面板中的 API 密钥和已验证的发送者邮箱地址

或者使用 community_contributions/2_lab2_with_resend_email 中的 "Resend Email" 替代方案

（或者 - 你也可以将下面的邮件发送代码替换为 Pushover 调用，或者简单地写入文件）

## 步骤1：Agent 工作流

In [ ]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [ ]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="gpt-4o-mini"
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="gpt-4o-mini"
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="gpt-4o-mini"
)

In [ ]:

result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

In [ ]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")


In [ ]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model="gpt-4o-mini"
)

In [ ]:
message = "Write a cold sales email"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")


现在去查看追踪记录：

https://platform.openai.com/traces

## 第二部分：使用工具

现在我们将加入工具。

还记得那些繁琐的 json 样板代码和带有 if 逻辑的 `handle_tool_calls()` 函数吗...

In [ ]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="gpt-4o-mini",
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="gpt-4o-mini",
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="gpt-4o-mini",
)

In [ ]:
sales_agent1

## 步骤2和3：工具与 Agent 交互

还记得那些繁琐的 json 样板代码吗？

只需用 `@function_tool` 装饰器包装你的函数即可

In [ ]:
@function_tool
def send_email(body: str):
    """ 将给定内容的邮件发送给所有销售潜在客户 """
    resend.api_key = 're_QTqQh9Yx_6ceG31HmFphEhpSncDhgBrZh'
    params = {
        'from': 'zzh@ztest.online',
        'to': ['443583917@qq.com'],
        'subject': '销售邮件',
        'text': body,
    }
    r = resend.Emails.send(params)
    return {"status": "success", "id": r.get("id")}

### 这已自动转换为工具，json 样板代码已自动生成

In [ ]:
# 让我们看看它
send_email

### 你也可以将 Agent 转换为工具

In [ ]:
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")
tool1

### 现在我们可以将所有工具汇集在一起：

3个邮件撰写 Agent 各自作为一个工具

再加上我们发送邮件的函数工具

In [ ]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

tools

## 现在轮到我们的销售经理 - 规划 Agent

In [ ]:
# 改进后的指令，感谢学生 Guillermo F.

instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""


sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model="gpt-4o-mini")

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">等等 - 你没收到邮件？？</h2>
            <span style="color:#ff7800;">非常感谢学生 Chris S. 描述了问题和修复方法。 
            如果在运行上一个单元格后没有收到邮件，请检查以下几点：<br/>
            首先，检查垃圾邮件文件夹！多位学生发现邮件被归类到垃圾邮件中！<br/>其次，print(result) 看看是否有 SSL 相关错误。
            如果有 SSL 错误，请查看这些<a href="https://chatgpt.com/share/680620ec-3b30-8012-8c26-ca86693d0e3d">网络技巧</a>并查看下一个单元格中的说明。也可以在 OpenAI 中查看追踪记录，并在 SendGrid 网站上排查线索。如果需要帮助，请告诉我！
            </span>
        </td>
    </tr>
</table>

### 学生 Oleksandr 在 Windows 11 上发送邮件的额外建议：

如果遇到 SSL 证书错误：  
在终端运行：`uv pip install --upgrade certifi`

然后运行以下代码：
```python
import certifi
import os
os.environ['SSL_CERT_FILE'] = certifi.where()
```

感谢 Oleksandr！

## 记得查看追踪记录

https://platform.openai.com/traces

然后检查你的邮箱！！

### Handoffs 代表一种 Agent 委托给另一个 Agent 的方式，将控制权传递过去

Handoffs 和 Agents-as-tools 类似：

两者都允许 Agent 与另一个 Agent 协作

使用工具时，控制权会返回

使用 Handoffs 时，控制权会转移过去

In [ ]:

subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")


In [ ]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ 将给定主题和 HTML 内容的邮件发送给所有销售潜在客户 """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("ed@edwarddonner.com")  # 改为你已验证的发送者邮箱
    to_email = To("ed.donner@gmail.com")  # 改为你的收件人邮箱
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [ ]:
tools = [subject_tool, html_tool, send_html_email]

In [ ]:
tools

In [ ]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it")


### 现在我们有了3个工具和1个 handoff

In [ ]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]
print(tools)
print(handoffs)

In [ ]:
# 改进后的指令，感谢学生 Guillermo F.

sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini")

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

### 记得查看追踪记录

https://platform.openai.com/traces

然后检查你的邮箱！！

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">练习</h2>
            <span style="color:#ff7800;">你能识别出这里使用的 Agentic 设计模式吗？<br/>
            根据 Anthropic 的定义，哪一行代码将其从 Agentic "工作流" 变成了 "Agent"？<br/>
            尝试添加更多工具和 Agent！你可以添加处理邮件合并以发送到列表的工具。<br/><br/>
            高难度挑战：研究如何使用 SendGrid 在用户回复邮件时调用回调 webhook，
            然后让 SDR 响应以保持对话进行！这可能需要一些 "vibe coding" 😂
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">商业应用</h2>
            <span style="color:#00bfff;">这可以立即应用于销售自动化；但更广泛地说，这可以应用于通过对话和工具实现任何业务流程的端到端自动化。想想如何将这样的 Agent 解决方案应用到你日常工作中的场景。
            </span>
        </td>
    </tr>
</table>

## 补充说明：

Google 发布了他们的 Agent Development Kit (ADK)。虽然它还没有本课程中其他框架那样的影响力，但正在获得一些关注。值得注意的是，它看起来与 OpenAI Agents SDK 非常相似。以下是一段 ADK 示例代码预览：

```
root_agent = Agent(
    name="weather_time_agent",
    model="gemini-2.0-flash",
    description="Agent to answer questions about the time and weather in a city.",
    instruction="You are a helpful agent who can answer user questions about the time and weather in a city.",
    tools=[get_weather, get_current_time]
)
```

嗯，看起来很熟悉！

有学生在 community_contributions 中贡献了一个使用 ADK 的客户服务 Agent。